# **Retail Store Performance Evaluation**

This notebook

## **Data Source**
This project features a messy dataset generated by AI that will allow me to showcase my skills in data analytics.


---
**This project answers the questions:**
1. Profit VS Revenue
2. Which shows growth or decreasing trends?
3. Which categories are high-performing and loss performing?
4. Looks for products with high discount and low in profit
5. Which products are returned the most?
6. Which preoducts are worst-performing?
7. Why is profit declining despite steady sales?


---
## **NOTES**
1. This notebook uses pandas version 2.3.1. A FutureWarning may appear regarding downcasting behavior in .replace(), but it does not affect the output. Code has been updated to reflect future compatibility.
2. This notebook uses Matplotlib 3.8. A minor deprecation warning may appear regarding parameter naming, but it does not affect output.

In [ ]:
# Uploading of dataset
from google.colab import files
uploaded = files.upload()


Saving customers_raw_large.csv to customers_raw_large.csv
Saving orders_raw_large.csv to orders_raw_large.csv
Saving products_raw_large.csv to products_raw_large.csv


In [ ]:
# Importing of libraries and downloading of data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

customers = pd.read_csv("customers_raw_large.csv")
products = pd.read_csv("products_raw_large.csv")
orders = pd.read_csv("orders_raw_large.csv")

In [ ]:
from dateutil import parser

def parse_date(x):
    try:
        return parser.parse(str(x))
    except:
        return pd.NaT

customers['signup_date'] = customers['signup_date'].apply(parse_date)
orders['order_date'] = orders['order_date'].apply(parse_date)

## **Data Exploration & Checking of Quality Issues**

In [ ]:

customers.head()
products.head()
orders.head()

,order_id,order_date,customer_id,product_id,quantity,discount_pct,payment_method,returned
0,O001,2023-04-26,C019,P038,4,110%,Debit card,Yes
1,O002,2023-02-10,C120,P041,2,0.05,Debit card,No
2,O003,2023-05-08,C064,NaN,4,110%,Debit card,Yes
3,O004,2023-02-25,C007,P064,-2,5%,Debit Card,Yes
4,O005,2023-01-28,C018,P094,3,5%,Debit Card,Yes


In [ ]:
customers.info()
products.info()
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  125 non-null    object        
 1   full_name    125 non-null    object        
 2   gender       125 non-null    object        
 3   age          118 non-null    float64       
 4   city         125 non-null    object        
 5   state        125 non-null    object        
 6   signup_date  125 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 7.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    120 non-null    object
 1   product_name  120 non-null    object
 2   category      97 non-null     object
 3   price         120 non-null    int64 
 4   cost          120 non-n

In [ ]:
customers.isnull().sum()
products.isnull().sum()
orders.isnull().sum()

,0
order_id,0
order_date,0
customer_id,0
product_id,7
quantity,0
discount_pct,0
payment_method,0
returned,0


In [ ]:
# Duplicates
customers.duplicated().sum()
products.duplicated().sum()
orders.duplicated().sum()

# Inconsistencies
customers['gender'].unique()
products['category'].unique()
orders['payment_method'].unique()
orders['discount_pct'].unique()

array(['110%', '0.05', '5%', '0.10', '0'], dtype=object)

## **Data Cleaning: Ensuring Quality of Data**

In [ ]:
# Clean Customers table

# Remove Duplicates
customers.drop_duplicates()

# Fix names
customers['full_name'] = customers['full_name'].str.title()

# Standardize Gender
customers['gender'] = customers['gender'].replace({'M': 'Male', 'F': 'Female'})

# Fix city casing
customers['city'] = customers['city'].str.title()

# Fix invaliud ages
customers.loc[(customers['age'] < 0) | (customers['age'] > 100), 'age'] = np.nan

# Fix missing ages
customers['age'] = customers['age'].fillna(customers['age'].median())

# Fix date format
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

In [ ]:
print(customers)

    customer_id           full_name  gender   age       city state signup_date
0          C001        Noah Johnson  Female  65.0    Seattle    WA  2022-07-13
1          C002        Evelyn Allen    Male  50.0    Atlanta    GA  2021-07-23
2          C003    Charlotte Thomas    Male  18.0    Chicago    IL  2021-12-15
3          C004        Ava Anderson    Male  42.0    Seattle    WA  2021-12-19
4          C005      Isabella Smith    Male  42.0    Seattle    WA  2023-04-30
..          ...                 ...     ...   ...        ...   ...         ...
120        C096       Amelia Wright  Female  50.0    Chicago    IL  2021-10-10
121        C055        Amelia Brown  Female  38.0  San Diego    CA  2021-06-04
122        C060            Ava Hill  Female  60.0     Austin    TX  2022-03-15
123        C118  Benjamin Hernandez  Female  26.0    Phoenix    AZ  2022-11-18
124        C078     Benjamin Wilson  Female  18.0    Atlanta    GA  2022-05-06

[125 rows x 7 columns]


In [ ]:
# Clean products table

# Trim spaces
products['product_name'] = products['product_name'].str.strip()
products['supplier'] = products['supplier'].str.strip()

# Normalize case and strip spaces
products['category'] = products['category'].str.strip().str.lower()

# Replace variants
products['category'] = products['category'].replace({
    'electronic': 'electronics',
    'electronics': 'electronics'
})

# Standardize category
products['category'] = products['category'].str.title()

products['category'] = products['category'].replace({'Electronic': 'Electronics'})

# Handle missing category
products['category'] = products['category'].replace('', 'Unknown')
products['category'] = products['category'].fillna('Unknown')

# Fix negative cost
products.loc[products['cost'] < 0, 'cost'] = np.nan

# Fill missing cost
products['cost'] = products['cost'].fillna(products['cost'].median())

# Remove invalid rows
products = products[products['price'] > products['cost']]

In [ ]:
print(products)

    product_id product_name     category  price   cost      supplier
0         P001    Product 1  Accessories    203  183.0       ZenFlex
1         P002    Product 2      Unknown    252  224.0  SoundPro Ltd
2         P003    Product 3      Apparel    144  107.0       ZenFlex
3         P004    Product 4      Unknown    126   66.0       ZenFlex
5         P006    Product 6  Accessories    148   99.0  SoundPro Ltd
..         ...          ...          ...    ...    ...           ...
115       P116  Product 116      Apparel    267  200.0      SunStyle
116       P117  Product 117  Electronics     50    5.0       PureSip
117       P118  Product 118      Apparel     93    8.0     ClickTech
118       P119  Product 119      Unknown    196   89.0     ClickTech
119       P120  Product 120  Electronics    183  117.0    UrbanCarry

[97 rows x 6 columns]


In [ ]:
products['category'].unique()
products['category'].apply(lambda x: repr(x)).unique()

array(["'Accessories'", "'Unknown'", "'Apparel'", "'Sports'",
       "'Electronics'"], dtype=object)

In [ ]:
# Clean Orders table

# Remove Duplicates
orders = orders.drop_duplicates()

# Clean discount
orders['discount_pct'] = orders['discount_pct'].astype(str).str.replace('%', '')

orders['discount_pct'] = pd.to_numeric(orders['discount_pct'], errors='coerce')

# Convert percentages
orders.loc[orders['discount_pct'] > 1, 'discount_pct'] /= 100

# Remove invalid discounts
orders = orders[orders['discount_pct'] <= 1]

# Fix quantity
orders.loc[orders['quantity'] < 0, 'quantity'] = np.nan
orders['quantity'] = orders['quantity'].fillna(orders['quantity'].median())

# Standardize payment
orders['payment_method'] = orders['payment_method'].str.title()

# Fix date
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

# Remove missing product_id
orders = orders.dropna(subset=['product_id'])


In [ ]:
print(orders)

    order_id order_date customer_id product_id  quantity  discount_pct  \
1       O002 2023-02-10        C120       P041       2.0          0.05   
3       O004 2023-02-25        C007       P064       2.0          0.05   
4       O005 2023-01-28        C018       P094       3.0          0.05   
5       O006 2023-04-13        C073       P047       2.0          0.10   
7       O008 2023-01-25        C048       P044       2.0          0.10   
..       ...        ...         ...        ...       ...           ...   
143     O144 2023-04-22        C059       P008       3.0          0.05   
145     O146 2023-04-06        C096       P035       1.0          0.00   
146     O147 2023-03-11        C034       P074       1.0          0.05   
147     O148 2023-01-15        C110       P048       2.0          0.10   
148     O149 2023-02-28        C006       P002       2.0          0.00   

    payment_method returned  
1       Debit Card       No  
3       Debit Card      Yes  
4       Debit Card   

In [ ]:
# Validate cleaned data

customers.info()
products.info()
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  125 non-null    object        
 1   full_name    125 non-null    object        
 2   gender       125 non-null    object        
 3   age          125 non-null    float64       
 4   city         125 non-null    object        
 5   state        125 non-null    object        
 6   signup_date  125 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 7.0+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 97 entries, 0 to 119
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    97 non-null     object 
 1   product_name  97 non-null     object 
 2   category      97 non-null     object 
 3   price         97 non-null     int64  
 4   cost          97 non-nu

In [ ]:
# Checking
customers['signup_date'].isnull().sum()
orders['order_date'].isnull().sum()
orders['quantity'].isnull().sum()

np.int64(0)

## **Analysis & Business Insights**

In [ ]:
# Merging of Datasets

df = orders.merge(customers, on='customer_id', how='left').merge(products, on='product_id', how='left')

In [ ]:
# Creating Business Metrics

df['revenue'] = df['price'] * df['quantity'] * (1 - df['discount_pct'])
df['profit'] = df['revenue'] - df['cost'] * df['quantity']

## **Core Analysis**

### Total Revenue & Profit


This allows us to compare if the profit is low compared to revenue.

In [ ]:
# Total Revenue and Profit

name = "Products Summary"
total_revenue = float(df['revenue'].sum())
total_profit = float(df['profit'].sum())

summary = {
    "Name": name,
    "Total Revenue": total_revenue,
    "Total Profit": total_profit
}

print(summary)


{'Name': 'Products Summary', 'Total Revenue': 40995.549999999996, 'Total Profit': 15059.55}


### Monthly Revenue Trend


This allows us to for growth or decreasing trends.

In [ ]:
# Monthly Revenue Trend

df['month'] = df['order_date'].dt.to_period('M')
df.groupby('month')['revenue'].sum()

,revenue
month,
2023-01,4316.75
2023-02,4636.25
2023-03,8327.10
2023-04,9056.25
2023-05,6114.95
2023-06,8544.25


### Profit by Category


This allows us to see the High-performing categories and the loss-making categories with Sports category being the lowest in Profit and Accessories being the highest.

In [ ]:
# Profit by Category

df.groupby('category')['profit'].sum().sort_values()

,profit
category,
Sports,1230.50
Apparel,1438.75
Unknown,2040.40
Electronics,5128.45
Accessories,5221.45


### Discount Impact


This displays that that those with 0 to low discounts have a high profit. Which would indicate that the discount given affects our profit and revenue.

In [ ]:
# Discount Impact

df.groupby('discount_pct').mean(numeric_only=True).sort_index()


,quantity,age,price,cost,revenue,profit
discount_pct,,,,,,
0.00,2.269231,43.769231,205.363636,132.386364,492.272727,174.636364
0.05,2.196721,46.852459,180.872727,104.809091,378.773636,155.528182
0.10,2.322581,48.129032,176.333333,112.125000,388.875000,110.979167


### Return Analysis


This allows us to identify which categories have been returned the most. Which reveals that Electronics was the most returned in category, and Sports was least returned.

In [ ]:
# Return Analysis

df['returned'].value_counts(normalize=True)

,proportion
returned,
Yes,0.508475
No,0.491525


In [ ]:
df.groupby('category')['returned'].value_counts(normalize=True)

category     returned
Accessories  No          0.588235
             Yes         0.411765
Apparel      Yes         0.555556
             No          0.444444
Electronics  Yes         0.592593
             No          0.407407
Sports       No          0.700000
             Yes         0.300000
Unknown      Yes         0.571429
             No          0.428571
Name: proportion, dtype: float64

### Customer Insights

VIP Customers



Revenue Concentration

In [ ]:
# Customer Insights

df.groupby('customer_id')['revenue'].sum().sort_values(ascending=False).head(5)


,revenue
customer_id,
C060,2501.00
C022,2478.55
C093,2066.60
C018,2042.50
C078,2037.60


### Product-Level Profit


This analysis highlights the ten lowest-performing products, with the top four: Product 107, Product 115, Product 14, Product 32, and Product 57,  showing critically low profits in the single-digit range.

In [ ]:
df.groupby('product_name')['profit'].sum().sort_values().head(10)

,profit
product_name,
Product 107,0.40
Product 115,0.70
Product 14,6.00
Product 32,7.20
Product 57,13.55
Product 59,40.60
Product 45,43.00
Product 28,44.55
Product 94,44.85


### **Why is profit declining despite steady sales?**

Based on our analysis, we can conclude that:


1.   Products that have higher discount rate have lower profit.
2.   Most returned items are electronics, which may indicate that there have been defects reason why it's the most returned item.
3.   Even though Electronics has 59% return rate, it's still 2nd in most profitable products.
4.   Sports has only 30% return rate, but has the lowest profit ammong all categories.



### Insights & Possible Reasons:

*   It is highly possible that **Sports** category has the highest discount rate amongst all categories that resulted to low profit return. This needs to be evaluated if the discount percentage are appropriate in these items.  
*   Although Electronics has 2nd highest profit return, we still need to investigate why it has has 59% return rate. This will help boost sales even more if we are able to have control over the return rate of this product.
*   Discount Percentage: Evaluate discount rate to balance it across the products to avoid loss in profit.



In [ ]:
# Download final data

df.to_csv("final_retail_dataset.csv", index=False)

from google.colab import files
files.download("final_retail_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>